In [9]:
from transformers import TFBertTokenizer, TFBertForSequenceClassification
from sklearn.model_selection import train_test_split
import tensorflow as tf

# Load pre-trained BERT model and tokenizer (TensorFlow version)
model_name = "bert-base-uncased"
tokenizer = TFBertTokenizer.from_pretrained(model_name)
model = TFBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize the data
def tokenize_data(texts, labels, max_length=128):
    # Tokenize texts
    encodings = tokenizer(
        texts.tolist(),  # Convert pandas Series to list
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="tf",  # Return TensorFlow tensors
    )
    # Convert labels to TensorFlow tensor
    labels = tf.convert_to_tensor(labels.tolist())
    return encodings, labels

# Prepare dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)
train_encodings, train_labels = tokenize_data(train_texts, train_labels)
val_encodings, val_labels = tokenize_data(val_texts, val_labels)

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), train_labels))
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encodings), val_labels))

# Batch the datasets
train_dataset = train_dataset.shuffle(1000).batch(16)
val_dataset = val_dataset.batch(16)

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy("accuracy")
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# Train the model
model.fit(train_dataset, epochs=3, validation_data=val_dataset)

# Evaluate the model
model.evaluate(val_dataset)

ImportError: 
TFBertTokenizer requires the tensorflow_text library but it was not found in your environment. You can install it with pip as
explained here: https://www.tensorflow.org/text/guide/tf_text_intro.
Please note that you may need to restart your runtime after installation.
